In [ ]:
# 0) setup: session clock, embedded code, helpers
import base64, glob, hashlib, io, json, os, shlex, subprocess, tarfile, threading, time
T0 = time.time()
DEADLINE = T0 + 10.4 * 3600
CODE, OUT = '/kaggle/working/code', '/kaggle/working/out'
B64 = '...'
raw = base64.b64decode(B64)
assert hashlib.sha256(raw).hexdigest() == '2133d3b8d97801633f537eb0a3c31291bb27f7d8e441e7559ae3df99d8ea6f00'
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall(CODE)
os.makedirs(OUT, exist_ok=True)
print('code sha256 2133d3b8d97801633f537eb0a3c31291bb27f7d8e441e7559ae3df99d8ea6f00 from git 4bedf33+local changes')


def run(cmd, env=None, prefix=''):
    """Run a shell command in CODE and stream its output; return the exit code."""
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, **(env or {})))
    for line in p.stdout:
        print(prefix + line, end='', flush=True)
    rc = p.wait()
    print(f'{prefix}[exit {rc} after {(time.time() - T0) / 3600:.2f} h]', flush=True)
    return rc


subprocess.run('nvidia-smi --query-gpu=index,name,memory.total --format=csv; free -g | head -2; '
               r'find /kaggle/input -maxdepth 6 \( -name "*.mat" -o -name "best_ema.pt" \) | head', shell=True)
MAT = sorted(glob.glob('/kaggle/input/**/Pavia.mat', recursive=True))[0]
INIT = [p for p in glob.glob('/kaggle/input/**/best_ema.pt', recursive=True) if 'puformer' in p][0]
print('Pavia:', MAT, '| transfer init:', INIT)


In [ ]:
# 1) self-checks and a short CPU smoke run of the Pavia pipeline
assert run('python selfcheck.py') == 0
assert run('CUDA_VISIBLE_DEVICES= python train.py --dataset pavia --mat x --smoke --iters 6 --eval_every 3 '
           '--width 16 --stages 2 --bs 2 --amp 0 --q2n 1 --out /tmp/smoke') == 0


In [ ]:
# 2) two runs in parallel until the deadline: A from scratch (GPU 0), B transfer from Chikusei (GPU 1)
COMMON = (f'python train.py --dataset pavia --mat {shlex.quote(MAT)} --pad reflect --bs 8 --warmup 1000 --w_sam 0.05 '
          f'--w_ssim 0.1 --eval_every 1000 --log_every 1000 --deadline {DEADLINE}')
jobs = {'A': (f'{COMMON} --lr 3e-4 --out {OUT}/pavia_scratch', '0'),
        'B': (f'{COMMON} --lr 1.5e-4 --init_ckpt {shlex.quote(INIT)} --init_partial 1 --out {OUT}/pavia_transfer', '1')}
codes = {}
threads = [threading.Thread(target=lambda k=k, c=c, g=g: codes.__setitem__(k, run(c, {'CUDA_VISIBLE_DEVICES': g}, f'[{k}] ')))
           for k, (c, g) in jobs.items()]
[t.start() for t in threads]
[t.join() for t in threads]
print('exit codes:', codes)


In [ ]:
# 3) summary against TIP'26 Table IV (Pavia Center, reported values)
tip = dict(PSNR=47.1597, SSIM=0.9972, SAM=1.4542, ERGAS=0.8262, Q2n=0.9980, CC=0.9985, SCC=0.9976, RMSE_DN=35.7086)
for run_name in ('pavia_scratch', 'pavia_transfer'):
    f = f'{OUT}/{run_name}/results.json'
    if not os.path.exists(f):
        print(run_name, 'no results'); continue
    r = json.load(open(f))
    print(run_name, {k: r[k] for k in ('iters', 'epochs', 'best_val_PSNR', 'train_hours', 'dn_scale') if k in r})
    for k in ('test', 'test_tta', 'gsa_test', 'bicubic_test'):
        print(' ', k, {m: round(v, 4) for m, v in r[k].items()})
    t = r['test_tta']
    print('  vs TIP26 best:', {m: ('WIN' if (t[m] < v if m in ('SAM', 'ERGAS', 'RMSE_DN') else t[m] > v) else 'lose') + f' {t[m]:.4f}/{v}'
                                for m, v in tip.items() if m in t}, '| SSIM_psrt', round(t['SSIM_psrt'], 4))
subprocess.run(f'ls -la {OUT}/*; du -sh {OUT}', shell=True)
